# Interactive Sales Dashboard
**Week 6: Data Visualization Mastery with Seaborn & Plotly**

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.ticker as mticker, seaborn as sns
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
PALETTE=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B3']
PRODUCT_CLR=dict(zip(['Laptop','Phone','Tablet','Headphones','Monitor'],PALETTE))
REGION_CLR={'East':'#4C72B0','West':'#DD8452','North':'#55A868','South':'#C44E52'}
sns.set_theme(style='whitegrid',palette=PALETTE,font_scale=1.15)
plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False})
print('Libraries loaded')

## Day 1 — Load Data

In [ ]:
df=pd.read_csv('sales_data.csv',parse_dates=['Date'])
df['Month']=df['Date'].dt.to_period('M').dt.to_timestamp()
df['Revenue']=df['Total_Sales']
print(df.shape); df.head()

## Day 2 — Box & Violin Plots

In [ ]:
fig,ax=plt.subplots(figsize=(10,6))
order=df.groupby('Product')['Price'].median().sort_values(ascending=False).index
sns.boxplot(data=df,x='Product',y='Price',order=order,palette=PALETTE,width=0.55,ax=ax)
sns.stripplot(data=df,x='Product',y='Price',order=order,color='#222',size=3.5,alpha=0.35,jitter=True,ax=ax)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'Rs{x:,.0f}'))
ax.set_title('Price Distribution by Product',fontsize=15,fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(10,6))
order=df.groupby('Region')['Revenue'].median().sort_values(ascending=False).index
sns.violinplot(data=df,x='Region',y='Revenue',order=order,
               palette=list(REGION_CLR.values()),inner='quartile',linewidth=1.3,ax=ax)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'Rs{x/1e3:,.0f}K'))
ax.set_title('Revenue Distribution by Region',fontsize=15,fontweight='bold')
plt.tight_layout(); plt.show()

## Day 3 — Heatmaps

In [ ]:
corr=df[['Quantity','Price','Total_Sales']].corr()
fig,ax=plt.subplots(figsize=(7,5))
sns.heatmap(corr,annot=True,fmt='.2f',cmap='RdYlGn',vmin=-1,vmax=1,
            linewidths=0.5,annot_kws={'size':13,'weight':'bold'},ax=ax)
ax.set_title('Correlation Matrix',fontsize=15,fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
pivot=df.pivot_table(values='Revenue',index='Product',columns='Region',aggfunc='sum')
fig,ax=plt.subplots(figsize=(9,5))
sns.heatmap(pivot,annot=True,fmt=',.0f',cmap='YlOrRd',linewidths=0.4,annot_kws={'size':10},ax=ax)
ax.set_title('Total Revenue: Product x Region',fontsize=15,fontweight='bold')
plt.tight_layout(); plt.show()

## Day 4 — Multi-Plot Dashboard

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Sales Overview',fontsize=18,fontweight='bold')
rev_prod=df.groupby('Product')['Revenue'].sum().sort_values(ascending=False)
axes[0,0].bar(rev_prod.index,rev_prod.values/1e6,color=PALETTE,edgecolor='white')
axes[0,0].set_title('Revenue by Product (M)',fontweight='bold')
rev_reg=df.groupby('Region')['Revenue'].sum().sort_values()
axes[0,1].barh(rev_reg.index,rev_reg.values/1e6,
               color=[REGION_CLR[r] for r in rev_reg.index],edgecolor='white')
axes[0,1].set_title('Revenue by Region (M)',fontweight='bold')
monthly=df.groupby('Month')['Revenue'].sum().reset_index()
axes[1,0].plot(monthly['Month'],monthly['Revenue']/1e6,marker='o',color=PALETTE[0],linewidth=2.2)
axes[1,0].fill_between(monthly['Month'],monthly['Revenue']/1e6,alpha=0.15,color=PALETTE[0])
axes[1,0].set_title('Monthly Revenue Trend',fontweight='bold')
axes[1,0].tick_params(axis='x',rotation=30)
qty=df.groupby('Product')['Quantity'].sum()
axes[1,1].pie(qty.values,labels=qty.index,autopct='%1.1f%%',colors=PALETTE,startangle=140,
              wedgeprops=dict(edgecolor='white',linewidth=1.5))
axes[1,1].set_title('Units Sold by Product',fontweight='bold')
for ax in axes.flat: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## Day 5 — Interactive Plotly Charts

In [ ]:
mp=df.groupby(['Month','Product'])['Revenue'].sum().reset_index()
mp['Month_str']=mp['Month'].dt.strftime('%b %Y')
fig6=px.line(mp,x='Month_str',y='Revenue',color='Product',markers=True,
             color_discrete_map=PRODUCT_CLR,template='plotly_white',
             title='<b>Monthly Revenue Trend by Product</b>')
fig6.update_layout(hovermode='x unified',height=420)
fig6.show()

In [ ]:
agg=df.groupby(['Product','Region']).agg(
    Total_Revenue=('Revenue','sum'),Avg_Price=('Price','mean'),Total_Qty=('Quantity','sum')
).reset_index()
fig7=px.scatter(agg,x='Avg_Price',y='Total_Revenue',size='Total_Qty',
                color='Product',facet_col='Region',color_discrete_map=PRODUCT_CLR,
                template='plotly_white',title='<b>Revenue vs Avg Price</b>',size_max=45)
fig7.update_layout(height=420)
fig7.show()

In [ ]:
fig8=px.sunburst(df,path=['Region','Product'],values='Revenue',color='Region',
                 color_discrete_map=REGION_CLR,template='plotly_white',
                 title='<b>Customer Segmentation: Region to Product Revenue</b>')
fig8.update_traces(textinfo='label+percent entry')
fig8.update_layout(height=480)
fig8.show()

## Day 6 & 7 — Dashboard Integration & Summary

```bash
python dashboard.py        # generate all charts
python dashboard.py --show # open in browser
```

| Chart | Type | Key Insight |
|-------|------|-------------|
| Price Distribution | Box Plot | Laptops highest median, all categories have outliers |
| Revenue by Region | Violin | East widest spread; South most consistent |
| Correlation Matrix | Heatmap | Price & Qty both independently drive sales |
| Product x Region | Heatmap | Laptop dominates all regions |
| Overview Grid | 2x2 Subplots | Full summary view |
| Monthly Trend | Plotly Line | Feb peak; Laptops lead every month |
| Bubble Chart | Plotly Scatter | South Laptops best ROI per unit |
| Sunburst | Plotly Sunburst | East=28% revenue; Laptops=40% product share |
